# Colab Quickstart

Open this notebook in Colab (**Runtime -> Change runtime type -> GPU**) and run the cells top to bottom. The same code runs locally; Colab just gives you a free GPU for the VGG19 / MLP training steps.

**Why this exists:** the full Stage 2 + Stage 3 pipeline is already packaged under `src/` and `scripts/`. This notebook is a thin wrapper that clones the repo, mounts Drive (so you can point at your picture folder), and runs the pipeline. No code lives here - anything you need to change lives in `src/`.

## 1. Clone the repo and install

In [ ]:
GITHUB_URL = 'https://github.com/ekw4tu/DS3001-Project.git'

!git clone $GITHUB_URL /content/DS3001-Project 2>/dev/null || (cd /content/DS3001-Project && git pull)
%cd /content/DS3001-Project
!pip install -q -r requirements.txt
# Swap CPU onnxruntime for the GPU build so ArcFace uses CUDA on Colab
!pip uninstall -y -q onnxruntime && pip install -q onnxruntime-gpu

## 2. Mount Drive and point at your data folder

**Why:** this is how we access the gallery/probe images without re-uploading them every Colab session. The env var `FR_DATA_ROOT` overrides the default `data/` path so the pipeline reads straight from Drive.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['FR_DATA_ROOT'] = '/content/drive/MyDrive/DS_NN_Project_Pictures'
os.environ['FR_ARTIFACTS_DIR'] = '/content/drive/MyDrive/facial_recognition_artifacts'
!ls "$FR_DATA_ROOT"

## 3. Build the full embedding dataset + gallery

**Why one command:** replaces Stage 2 cells 7 (ArcFace loop) and 10 (VGG19 loop) plus the train/test split. Takes ~5-10 minutes on GPU.

In [ ]:
!python scripts/build_gallery.py --gpu

## 4. Run the full Stage 3 evaluation

**Why:** reproduces the unified classifier table from Stage 3 cell 10, plus the per-condition breakdown and the `identify()` validation. Single command, held-out test set only.

In [ ]:
!python scripts/evaluate.py

## 5. Add a new identity without rebuilding everything

**Why:** drop 5-10 photos of a new person into a folder, run one command, and the gallery now recognizes them. Takes seconds, not minutes.

In [ ]:
# Example: uncomment and run after adding a folder of photos.
# !python scripts/add_identity.py --name 'Alice Smith' --folder data/Gallery/aliceGallery --gpu

## 6. In-class challenge: identify unknown feature vectors

**Why:** the Stage 3 PDF section 3 says each team receives 5 ArcFace vectors and must match them to classmates. This is the exact script you run during the 4/21 or 4/23 session.

In [ ]:
# Save the vectors you receive as /content/unknowns.npy, then:
# !python scripts/identify_unknowns.py /content/unknowns.npy --top-k 5